[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/10_agent_v2.ipynb)

# 10. Agent v2

**Part 2: Tools** · about 30 minutes

Time to put Part 2 together. The agent from notebook 05 could only read files, rewrite them whole and run one fixed test command. The one we build here has:

| | Agent v1 (notebook 05) | Agent v2 (this notebook) |
|---|---|---|
| Reading | Whole files | `find`, plus line ranges |
| Changing code | Rewrite the whole file | `edit_file` for exact changes, `create_file` for new files |
| Running things | One fixed test command | Any shell command, with permissions |
| Safety | Workspace folder | Workspace, permission policy, resource limits, audit log |

And a harder job. Instead of "make the failing tests pass", it gets a **bug report**, the way bugs usually arrive:

> "Running `python wordcount.py empty.txt` crashes. Please fix it and add a test so it doesn't come back."

There's no failing test to point the way. The agent has to reproduce the crash, find the cause, fix it, prove the fix and write the test itself. Then we check its work in a way that's easy to skip: **does the new test actually catch the bug?**

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import difflib
import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. The project

A small command-line tool that reports word statistics. It works on normal files and crashes on an empty one. There are tests, and they all pass, because nobody tested the empty case. That's how most real bugs look.

In [3]:
WORDCOUNT = '''\
"""Print word statistics for a text file."""

import sys


def stats(text):
    words = text.split()
    total_letters = sum(len(word) for word in words)
    return {"words": len(words), "average_length": round(total_letters / len(words), 2)}


def main(path):
    with open(path) as f:
        result = stats(f.read())
    print(f"{result['words']} words, average length {result['average_length']}")


if __name__ == "__main__":
    main(sys.argv[1])
'''

TESTS = '''\
import unittest

from wordcount import stats


class StatsTests(unittest.TestCase):
    def test_counts_words(self):
        self.assertEqual(stats("one two three")["words"], 3)

    def test_average_length(self):
        self.assertEqual(stats("ab abcd")["average_length"], 3.0)


if __name__ == "__main__":
    unittest.main()
'''

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
(workspace / "wordcount.py").write_text(WORDCOUNT)
(workspace / "test_wordcount.py").write_text(TESTS)
(workspace / "empty.txt").write_text("")
(workspace / "poem.txt").write_text("the quick brown fox jumps over the lazy dog\n")

44

## 2. The toolbox

This is the code from notebooks 06 to 09, gathered in one place. Nothing here is new, so skim it if you've done those notebooks. If you jumped straight here, each piece is explained in the notebook named in its comment.

In [4]:
# --- Files (notebook 06) -----------------------------------------------------

def safe_path(path):
    full = (workspace / path).resolve()
    if not full.is_relative_to(workspace):
        raise PermissionError(f"{path!r} is outside the workspace")
    return full


def read_file(path, start=1, end=None):
    lines = safe_path(path).read_text().splitlines()
    end = len(lines) if end is None else min(end, len(lines))
    if not 1 <= start <= max(len(lines), 1):
        return f"Error: start must be between 1 and {len(lines)}"
    shown = "\n".join(f"{n:>4} | {line}" for n, line in enumerate(lines[start - 1 : end], start))
    return f"{path}: lines {start}-{end} of {len(lines)}\n{shown}"


def find(pattern):
    hits = [f"{f.relative_to(workspace).as_posix()}:{n}: {line.strip()}"
            for f in sorted(workspace.rglob("*.py"))
            for n, line in enumerate(f.read_text().splitlines(), 1) if pattern in line]
    return "\n".join(hits[:50]) if hits else f"No matches for {pattern!r}"


def edit_file(path, old, new):
    file = safe_path(path)
    text = file.read_text()
    if old == new:
        return "Error: old and new are identical, so there is nothing to change."
    if re.search(r"^\s*\d+ \| ", old, re.MULTILINE):
        return "Error: old contains line numbers like '  12 | '. Copy only the code after the bar."
    count = text.count(old)
    if count == 0:
        first = old.strip().splitlines()[0].strip() if old.strip() else ""
        close = difflib.get_close_matches(first, [l.strip() for l in text.splitlines()], n=1, cutoff=0.6)
        return "Error: old was not found in the file." + (f" The closest line is: {close[0]!r}" if close else "")
    if count > 1:
        return f"Error: old appears {count} times. Include more surrounding lines so it matches exactly once."
    line = text[: text.index(old)].count("\n") + 1
    file.write_text(text.replace(old, new, 1))
    return f"Edited {path}.\n" + read_file(path, max(1, line - 2), line + new.count("\n") + 2)


def create_file(path, content):
    """New files only. Changing an existing file must go through edit_file."""
    file = safe_path(path)
    if file.exists():
        return f"Error: {path} already exists. Use edit_file to change it."
    file.parent.mkdir(parents=True, exist_ok=True)
    file.write_text(content)
    return f"Created {path} ({len(content.splitlines())} lines)."

`create_file` is new, and the important part is the refusal. In notebook 05 one `write_file` both created and overwrote files, which meant one wrong path could wipe out real code. Now the only way to change an existing file is a precise edit.

In [5]:
# --- Commands: permissions (notebook 08) -----------------------------------------

OPERATORS = {";", "&&", "||", "|", "&", ">", ">>", "<", "(", ")"}
READ_ONLY = {"ls", "cat", "head", "tail", "wc", "grep", "find", "pwd", "echo", "diff"}
NEVER = {"sudo", "su", "curl", "wget", "ssh", "scp", "nc", "dd", "mkfs", "shutdown", "reboot", "chmod", "chown"}


def tokenize(command):
    lexer = shlex.shlex(command, posix=True, punctuation_chars=True)
    lexer.whitespace_split = True
    return list(lexer)


def decide(command):
    try:
        tokens = tokenize(command)
    except ValueError:
        return "ask", "couldn't parse the command"
    if not tokens:
        return "deny", "empty command"
    programs = [tokens[0]] + [tokens[i + 1] for i, t in enumerate(tokens[:-1]) if t in OPERATORS]
    names = {Path(p).name for p in programs}
    if names & NEVER:
        return "deny", f"{sorted(names & NEVER)[0]} is never allowed"
    if OPERATORS & set(tokens) or "`" in command or "$(" in command:
        return "ask", "compound command"
    program = Path(tokens[0]).name
    if program == "find" and {"-delete", "-exec", "-execdir", "-ok"} & set(tokens):
        return "ask", "find that changes or runs things"
    if program in READ_ONLY:
        return "allow", "read-only command"
    if tokens[:3] == ["python", "-m", "unittest"]:
        return "allow", "running the tests"
    # New in v2: running a script from the project is allowed, because the sandbox below contains it.
    if program in ("python", "python3") and len(tokens) > 1 and tokens[1].endswith(".py"):
        try:
            safe_path(tokens[1])
            return "allow", "project script, sandboxed"
        except PermissionError:
            return "deny", "script outside the project"
    return "ask", f"{program} isn't on the allow list"

One rule changed from notebook 08, and it's worth stopping on. **Running a project script is now allowed.** In notebook 08 that would have been reckless, because the script could do anything. With the sandbox underneath, a runaway script gets stopped, a disk-filling one gets cut off, and our keys aren't in its environment. Stronger isolation is what lets you loosen permissions safely. Real agents work the same way: the better the sandbox, the less often they need to ask you.

In [6]:
# --- Commands: sandbox (notebooks 07 and 09) -----------------------------------------

HAS_LIMITS = os.name == "posix"
if HAS_LIMITS:
    import resource

PYTHON = shlex.quote(sys.executable)


def apply_limits():
    resource.setrlimit(resource.RLIMIT_CPU, (10, 11))
    resource.setrlimit(resource.RLIMIT_FSIZE, (20 * 1024 * 1024,) * 2)
    if sys.platform.startswith("linux"):
        resource.setrlimit(resource.RLIMIT_AS, (1024 * 1024 * 1024,) * 2)


def describe(code):
    if code is None:
        return "stopped: took longer than the time limit"
    if code < 0 or code > 128:
        try:
            return f"stopped by signal {signal.Signals(-code if code < 0 else code - 128).name}"
        except ValueError:
            pass
    return f"exit code: {code}"


def run_sandboxed(command, timeout=30):
    command = re.sub(r"(?<![\w./-])python3?(?=\s|$)", lambda _: PYTHON, command)
    env = {k: os.environ[k] for k in ("PATH", "SYSTEMROOT", "LANG") if k in os.environ}
    env["HOME"] = str(workspace)
    options = dict(shell=True, cwd=workspace, env=env, text=True,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if HAS_LIMITS:
        options.update(start_new_session=True, preexec_fn=apply_limits)
    process = subprocess.Popen(command, **options)
    try:
        output, _ = process.communicate(timeout=timeout)
        code = process.returncode
    except subprocess.TimeoutExpired:
        if HAS_LIMITS:
            os.killpg(process.pid, signal.SIGKILL)
        else:
            subprocess.run(["taskkill", "/T", "/F", "/PID", str(process.pid)], capture_output=True)
        output, _ = process.communicate()
        code = None
    output = output.replace(str(workspace), ".")
    if len(output) > 4000:
        output = f"{output[:2000]}\n[... {len(output) - 4000:,} characters cut ...]\n{output[-2000:]}"
    return f"{describe(code)}\n{output}"


# --- Putting permissions and sandbox together -------------------------------------------

INTERACTIVE = False  # set to True to approve "ask" commands yourself
AUDIT = []


def approve(command, reason):
    if INTERACTIVE:
        return input(f"\nAllow `{command}`? ({reason}) [y/N] ").strip().lower() == "y"
    print(f"    approval needed ({reason}). Nobody is here to approve, so: denied.")
    return False


def run_command(command):
    decision, reason = decide(command)
    if decision == "ask":
        decision = "allow" if approve(command, reason) else "user said no"
    AUDIT.append({"command": command, "decision": decision, "reason": reason})
    if decision == "deny":
        return f"Denied by policy: {reason}. Don't look for a way around this."
    if decision == "user said no":
        return "Denied: the user didn't approve this command. Don't retry it. Explain what you wanted to do."
    return run_sandboxed(command)

In [7]:
def tool(name, description, **parameters):
    properties, required = {}, []
    for key, spec in parameters.items():
        if isinstance(spec, tuple):  # (type, description) marks an optional parameter
            properties[key] = {"type": spec[0], "description": spec[1]}
        else:
            properties[key] = {"type": "string", "description": spec}
            required.append(key)
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required}}}


TOOLS = [
    tool("run_command", "Run a shell command in the project folder, sandboxed. Some commands need approval. "
                        "Output starts with the exit code. Use `python` to run Python.", command="The command"),
    tool("find", "Search the project's Python files for text. Returns file:line: text.", pattern="Text to find"),
    tool("read_file", "Read a file with line numbers, optionally only lines start to end.", path="File path",
         start=("integer", "First line (optional)"), end=("integer", "Last line (optional)")),
    tool("edit_file", "Replace text in an existing file. old must match exactly once, without line numbers.",
         path="File path", old="Exact text to replace", new="Replacement text"),
    tool("create_file", "Create a new file. Fails if the file exists.", path="File path", content="File content"),
]

REGISTRY = {"run_command": run_command, "find": find, "read_file": read_file,
            "edit_file": edit_file, "create_file": create_file}

print(f"{len(TOOLS)} tools:", ", ".join(REGISTRY))

5 tools: run_command, find, read_file, edit_file, create_file


## 3. The agent

The loop is the same as in notebook 05. The system prompt is where the working method lives, and this one is written like instructions for a new teammate: what to do, in what order, and what counts as finished.

In [8]:
SYSTEM_PROMPT = """You are a careful coding agent. You fix bugs in a small Python project.

How to work:
1. Reproduce the problem first by running the command from the bug report. Read the error.
2. Find and read the code involved. Read only the lines you need.
3. Fix it with edit_file, changing as little as possible.
4. Run the original command again to confirm the fix.
5. Add a test for the bug in a new file with create_file, then run all tests with `python -m unittest`.

You're done only when the original command works and all tests pass. Finish with a short summary of
the cause and the fix. Don't claim anything you didn't see in a tool result."""


def call_model(messages):
    payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def run_tool_call(call):
    name = call["function"]["name"]
    if name not in REGISTRY:
        return f"Error: unknown tool {name!r}. Available tools: {sorted(REGISTRY)}"
    try:
        return str(REGISTRY[name](**json.loads(call["function"]["arguments"] or "{}")))
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"


def run_agent(task, max_steps=20):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]
    stats = {"model_calls": 0, "tool_calls": 0, "input_tokens": 0, "errors": 0}
    for step in range(1, max_steps + 1):
        data = call_model(messages)
        message = data["choices"][0]["message"]
        messages.append(message)
        stats["model_calls"] += 1
        stats["input_tokens"] += (data.get("usage") or {}).get("prompt_tokens", 0)
        calls = message.get("tool_calls") or []
        if not calls:
            return {"status": "done", "summary": message.get("content"), **stats}
        for call in calls:
            result = run_tool_call(call)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            stats["tool_calls"] += 1
            stats["errors"] += result.startswith(("Error", "Denied"))
            arguments = call["function"]["arguments"] or ""
            print(f"[{step:>2}] {call['function']['name']}({arguments[:75]}{'...' if len(arguments) > 75 else ''})")
            print(f"     {result.splitlines()[0][:90] if result else '(empty)'}")
    return {"status": "max_steps", "summary": None, **stats}

## 4. Confirm the bug, then hand it over

Before the agent starts, we check the bug report ourselves, like any good engineer would.

In [9]:
print(run_sandboxed("python wordcount.py poem.txt"))
print(run_sandboxed("python wordcount.py empty.txt"))

exit code: 0
9 words, average length 3.89

exit code: 1
Traceback (most recent call last):
  File "./wordcount.py", line 19, in <module>
    main(sys.argv[1])
    ~~~~^^^^^^^^^^^^^
  File "./wordcount.py", line 14, in main
    result = stats(f.read())
  File "./wordcount.py", line 9, in stats
    return {"words": len(words), "average_length": round(total_letters / len(words), 2)}
                                                         ~~~~~~~~~~~~~~^~~~~~~~~~~~
ZeroDivisionError: division by zero



In [10]:
started = time.perf_counter()
result = run_agent(
    "Bug report: running `python wordcount.py empty.txt` crashes. "
    "Please fix it and add a test so it doesn't come back."
)
elapsed = time.perf_counter() - started

print()
print("Status: ", result["status"])
print("Summary:", result["summary"])

[ 1] run_command({"command": "python wordcount.py empty.txt"})
     exit code: 1


[ 2] read_file({"path": "wordcount.py"})
     wordcount.py: lines 1-19 of 19


[ 3] edit_file({"path": "wordcount.py", "old": "round(total_letters / len(words), 2)", "ne...)
     Edited wordcount.py.


[ 4] run_command({"command": "python wordcount.py empty.txt"})
     exit code: 0


[ 5] create_file({"path": "test_empty.py", "content": "import unittest\n\nfrom wordcount imp...)
     Created test_empty.py (12 lines).


[ 6] run_command({"command": "python -m unittest -q"})
     exit code: 0



Status:  done
Summary: wordcount.py divided by the number of words, which crashes on an empty file. It now reports an average of 0.0 when there are no words. I added test_empty.py for this case, and all tests pass.


## 5. Check everything, trust nothing

The model's summary is a claim. We check four things in code:

1. **The original command works.**
2. **All tests pass.**
3. **There are more tests than before**, so a test was really added.
4. **The new test catches the bug.** We put the original buggy code back, run the tests, and expect a failure. A test that passes on buggy code proves nothing. This is called a *mutation check*, and it catches a surprising number of "tests" written to pass rather than to test.

In [11]:
def count_tests(output):
    match = re.search(r"Ran (\d+) test", output)
    return int(match.group(1)) if match else 0


command_works = run_sandboxed("python wordcount.py empty.txt").startswith("exit code: 0")
tests_now = run_sandboxed("python -m unittest")
tests_pass = tests_now.startswith("exit code: 0")
tests_added = count_tests(tests_now) > 2

# The mutation check: restore the bug for a moment and make sure the tests notice.
fixed_source = (workspace / "wordcount.py").read_text()
(workspace / "wordcount.py").write_text(WORDCOUNT)
catches_bug = not run_sandboxed("python -m unittest").startswith("exit code: 0")
(workspace / "wordcount.py").write_text(fixed_source)

print("The fix:\n")
for line in difflib.unified_diff(WORDCOUNT.splitlines(), fixed_source.splitlines(),
                                 "wordcount.py (before)", "wordcount.py (after)", lineterm="", n=1):
    print(line)

print("\n--- Scorecard ---")
print(f"Original command works:     {command_works}")
print(f"All tests pass:             {tests_pass}  ({count_tests(tests_now)} tests, there were 2)")
print(f"New test catches the bug:   {catches_bug}")
print(f"Old test file untouched:    {(workspace / 'test_wordcount.py').read_text() == TESTS}")
print(f"Model calls / tool calls:   {result['model_calls']} / {result['tool_calls']}")
print(f"Tool errors or denials:     {result['errors']}")
print(f"Time:                       {elapsed:.1f}s" + (" (simulated)" if llm.offline else ""))

The fix:

--- wordcount.py (before)
+++ wordcount.py (after)
@@ -8,3 +8,3 @@
     total_letters = sum(len(word) for word in words)
-    return {"words": len(words), "average_length": round(total_letters / len(words), 2)}
+    return {"words": len(words), "average_length": round(total_letters / len(words), 2) if words else 0.0}
 

--- Scorecard ---
Original command works:     True
All tests pass:             True  (3 tests, there were 2)
New test catches the bug:   True
Old test file untouched:    True
Model calls / tool calls:   7 / 6
Tool errors or denials:     0
Time:                       3.3s (simulated)


In [12]:
# Checkpoint: the offline script has to pass every check, including the mutation check.
if llm.offline:
    assert command_works and tests_pass and tests_added and catches_bug
solved = command_works and tests_pass and tests_added and catches_bug
print("Checkpoint passed." if solved else "Not fully solved. The scorecard shows which part failed.")

Checkpoint passed.


### The audit log

Every command the agent asked for, and what the permission layer decided. When something goes wrong in a long run, this is where you start.

In [13]:
for entry in AUDIT:
    print(f"{entry['decision']:<13} {entry['command']}")

allow         python wordcount.py empty.txt
allow         python wordcount.py empty.txt
allow         python -m unittest -q


## 6. What you've built

Look back at notebook 01. A single HTTP request has become an agent that takes a bug report, reproduces the bug, fixes it with a precise edit, proves the fix, writes a test that really guards against it, and does all of that inside permissions and a sandbox.

There's still plenty it can't do:

| Problem | Example | Fixed in |
|---|---|---|
| The history grows forever | A big project overflows the context window | Part 3 |
| Substring search only | It can't find "where is this used?" | Part 3 |
| It forgets between runs | It relearns the project every time | Part 3 |
| One API error ends the run | A 429 or a timeout and it's over | Part 4 |
| No way to measure improvements | "Is the new prompt better?" is a guess | Part 4 |

## Exercises

1. Write a harder bug report. Make the crash happen only with a file that has just spaces in it. Does the agent's test cover that case?
2. Make the mutation check automatic. If the new test doesn't catch the bug, send the agent back with a message telling it so. How many rounds does your model need?
3. Set `INTERACTIVE = True` and act as the human. Approve a few commands and refuse others. How does the model react to a refusal?
4. Count the tokens used by notebook 05's agent and this one on a similar task. Where does the difference come from?

## Clean up

In [14]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.


## You finished Part 2

Your agent now has real tools and real limits. **Part 3, Context,** tackles what happens when the project is bigger than what the model can see at once: token budgets, compaction, memory and proper code search.

If this course helped you, a star on the repo helps other people find it.